# M3CoT LVAR Trace-Ablation And Raw-Trace Entropy Analysis

Analyze the seven LVAR self-evaluation runs produced by `run_scripts/run_m3cot_test_ablation_entropy_eval.sh`: five trace ablations and two untouched (`raw`) trace replays. Basic final-answer entropy is analyzed for all seven runs; step-hidden entropy and prefix-rollout analysis is restricted to the two raw runs.

`action_prefix_metrics` is recorded after each non-empty oracle decision block, not after a fixed eight-step loop. This is why the x-axis below is `prefix_step = block_idx + 1`: examples with fewer useful blocks simply stop contributing at later x positions. `examples` in the table is therefore a coverage count for each prefix step, and it should be read alongside accuracy and entropy so late-step averages are not mistaken for full-dataset averages.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

OUTPUT_ROOT = Path("outputs/inference/test_oracle_ablations")
VARIANT_ORDER = ["raw", "shuffled", "no_visual", "no_reasoning"]
EXPECTED_EXPERIMENTS = {
    ("global", "no_reasoning"),
    ("global", "shuffled"),
    ("global", "no_visual"),
    ("coarse", "shuffled"),
    ("coarse", "no_visual"),
    ("global", "raw"),
    ("coarse", "raw"),
}

def trace_context_from_output_path(path):
    match = re.search(r"_(global|coarse)_(?:raw|shuffled|no_visual|no_reasoning)", path.name)
    if not match:
        raise ValueError(f"Could not identify trace context from {path.name}")
    return match.group(1)

def replay_context_from_output_path(path, mining_context):
    match = re.search(r"_replayed-under_(global|coarse|full_context|global_mean)(?:_|\.)", path.name)
    return match.group(1) if match else mining_context

if not OUTPUT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Output directory not found: {OUTPUT_ROOT}. Run the ablation/entropy script first."
    )

## 1. Run Summary

Load all seven run summaries and compare final accuracy and trace length. Prefix-level entropy columns are populated only for the two raw runs.

The summary table is one row per completed replay run. `accuracy` is final-answer accuracy after replaying the trace and decoding an answer. `avg_trace_actions` is the average number of primitive actions inserted into the model state, so a decision block such as `PATCH_SEQ` can contribute several primitive actions. `avg_output_tokens` is the length of the decoded final answer. The entropy columns are uncertainty diagnostics: `step_hidden_entropy_mean` measures next-token LM uncertainty after each raw action-prefix insertion, while `prefix_rollout_*` decodes from intermediate prefixes and asks whether the partially replayed trace was already enough to answer correctly. These prefix columns are expected only for raw replays because the ablation script disables extra prefix instrumentation for the perturbed variants.

In [ ]:
summary_rows = []
for summary_path in sorted(OUTPUT_ROOT.glob("mined_by_lvar_ckpt/evaluated_by_lvar_ckpt/trace_variant_*/*_summary.json")):
    summary = json.loads(summary_path.read_text())
    metrics = summary.get("metrics", {})
    prefix = metrics.get("prefix_rollouts", {})
    trace_context = Path(summary["trace_path"]).stem.rsplit("_", 1)[-1]
    replay_context = replay_context_from_output_path(summary_path, trace_context)
    variant = summary.get("trace_variant", summary_path.parent.name.removeprefix("trace_variant_"))
    if (trace_context, variant) not in EXPECTED_EXPERIMENTS:
        continue
    summary_rows.append({
        "model": "lvar",
        "trace_context": trace_context,
        "replay_context": replay_context,
        "variant": variant,
        "total": metrics.get("total"),
        "correct": metrics.get("correct"),
        "accuracy": metrics.get("accuracy"),
        "avg_trace_actions": metrics.get("avg_trace_actions"),
        "avg_output_tokens": metrics.get("avg_output_tokens"),
        "step_hidden_entropy_mean": metrics.get("step_hidden_entropy_mean"),
        "step_hidden_entropy_count": metrics.get("step_hidden_entropy_count"),
        "prefix_rollout_accuracy": prefix.get("accuracy"),
        "prefix_rollout_count": prefix.get("total"),
        "prefix_token_entropy_mean": prefix.get("mean_decoded_token_entropy"),
        "prefix_answer_option_entropy_mean": prefix.get("mean_answer_option_entropy"),
        "summary_path": str(summary_path),
    })

if not summary_rows:
    raise RuntimeError(f"No run summaries found under {OUTPUT_ROOT}")
found = {(row["trace_context"], row["variant"]) for row in summary_rows}
missing = EXPECTED_EXPERIMENTS - found
if missing:
    print(f"Warning: missing experiment summaries: {sorted(missing)}")

summary_df = pd.DataFrame(summary_rows)
summary_df["variant"] = pd.Categorical(summary_df["variant"], VARIANT_ORDER, ordered=True)
summary_df = summary_df.sort_values(["model", "trace_context", "replay_context", "variant"]).reset_index(drop=True)
display(
    summary_df.drop(columns="summary_path").style
    .format({
        "accuracy": "{:.2%}",
        "prefix_rollout_accuracy": "{:.2%}",
        "step_hidden_entropy_mean": "{:.4f}",
        "prefix_token_entropy_mean": "{:.4f}",
        "prefix_answer_option_entropy_mean": "{:.4f}",
    }, na_rep="—")
    .hide(axis="index")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_df = summary_df.copy()
plot_df["run"] = (
    plot_df["model"].astype(str) + " / trace " + plot_df["trace_context"].astype(str)
    + " → replay " + plot_df["replay_context"].astype(str)
)

if HAS_SEABORN:
    sns.barplot(data=plot_df, x="variant", y="accuracy", hue="run", ax=axes[0])
    sns.barplot(data=plot_df, x="variant", y="avg_trace_actions", hue="run", ax=axes[1])
else:
    plot_df.pivot(index="variant", columns="run", values="accuracy").plot.bar(ax=axes[0])
    plot_df.pivot(index="variant", columns="run", values="avg_trace_actions").plot.bar(ax=axes[1])

axes[0].set(title="Final accuracy", ylabel="Accuracy", xlabel="Variant")
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].set(title="Average replayed trace length", ylabel="Primitive trace actions", xlabel="Variant")
for axis in axes:
    axis.tick_params(axis="x", rotation=20)
    legend = axis.get_legend()
    if legend is not None:
        legend.set_title("Model / context")
plt.tight_layout()
plt.show()

## 2. Raw-Trace Entropy And Accuracy Along The Action Prefix

Flatten `action_prefix_metrics` from only the untouched global and coarse trace replays. Ablation runs are excluded because prefix instrumentation was deliberately disabled for them.

In [ ]:
prefix_rows = []
for prediction_path in sorted(OUTPUT_ROOT.glob("mined_by_lvar_ckpt/evaluated_by_lvar_ckpt/trace_variant_raw/*.jsonl")):
    model = "lvar"
    variant = "raw"
    trace_context = trace_context_from_output_path(prediction_path)
    replay_context = replay_context_from_output_path(prediction_path, trace_context)
    with prediction_path.open() as handle:
        for line in handle:
            row = json.loads(line)
            for prefix in row.get("action_prefix_metrics", []):
                hidden = prefix.get("next_token_entropy") or {}
                rollout = prefix.get("rollout") or {}
                option_entropy = rollout.get("answer_option_entropy") or {}
                prefix_rows.append({
                    "model": model,
                    "trace_context": trace_context,
                    "replay_context": replay_context,
                    "variant": variant,
                    "example_id": row.get("example_id"),
                    "final_correct": bool(row.get("correct")),
                    "prefix_step": int(prefix.get("block_idx", 0)) + 1,
                    "action": prefix.get("action"),
                    "sequence_length": prefix.get("sequence_length"),
                    "next_token_entropy": hidden.get("entropy"),
                    "top_k_retained_mass": hidden.get("retained_probability_mass"),
                    "rollout_correct": rollout.get("correct"),
                    "rollout_token_entropy_mean": rollout.get("token_entropy_mean"),
                    "rollout_answer_option_entropy": option_entropy.get("entropy"),
                    "rollout_decoded_answer": rollout.get("decoded_answer"),
                })

prefix_df = pd.DataFrame(prefix_rows)
if prefix_df.empty:
    print("No raw-trace prefix metrics found. Confirm both raw runs completed successfully.")
else:
    prefix_df["variant"] = pd.Categorical(prefix_df["variant"], VARIANT_ORDER, ordered=True)
    prefix_summary = (
        prefix_df.groupby(["model", "trace_context", "replay_context", "variant", "prefix_step"], observed=True)
        .agg(
            examples=("example_id", "nunique"),
            next_token_entropy_mean=("next_token_entropy", "mean"),
            next_token_entropy_sem=("next_token_entropy", "sem"),
            rollout_accuracy=("rollout_correct", "mean"),
            rollout_token_entropy_mean=("rollout_token_entropy_mean", "mean"),
            rollout_answer_option_entropy_mean=("rollout_answer_option_entropy", "mean"),
        )
        .reset_index()
    )
    display(prefix_summary.head(30).style.format({
        "next_token_entropy_mean": "{:.4f}",
        "next_token_entropy_sem": "{:.4f}",
        "rollout_accuracy": "{:.2%}",
        "rollout_token_entropy_mean": "{:.4f}",
        "rollout_answer_option_entropy_mean": "{:.4f}",
    }, na_rep="—").hide(axis="index"))

In [ ]:
if not prefix_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
    axes = axes.ravel()
    prefix_plot = prefix_summary.copy()
    prefix_plot["run"] = (
        prefix_plot["model"].astype(str) + " / trace "
        + prefix_plot["trace_context"].astype(str) + " -> replay "
        + prefix_plot["replay_context"].astype(str)
    )
    metrics = [
        ("examples", "Examples contributing at prefix step", "Examples"),
        ("next_token_entropy_mean", "Next-token entropy after prefix insertion", "Entropy (nats)"),
        ("rollout_accuracy", "Accuracy when decoding from this prefix", "Accuracy"),
        ("rollout_answer_option_entropy_mean", "Answer-option entropy when decoding from this prefix", "Entropy (nats)"),
    ]
    for axis, (metric, title, ylabel) in zip(axes, metrics):
        if HAS_SEABORN:
            sns.lineplot(data=prefix_plot, x="prefix_step", y=metric, hue="run", marker="o", ax=axis)
        else:
            prefix_plot.pivot(index="prefix_step", columns="run", values=metric).plot(marker="o", ax=axis)
        axis.set(title=title, xlabel="Action-prefix step (decision block, not fixed loop step)", ylabel=ylabel)
        axis.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        if metric == "rollout_accuracy":
            axis.yaxis.set_major_formatter(PercentFormatter(1.0))
        legend = axis.get_legend()
        if legend is not None:
            legend.set_title("Run")
    plt.tight_layout()
    plt.show()


## 3. Final-Answer Entropy By Correctness For All Seven Runs

Use every run's ordinary entropy sidecar to compare final-answer uncertainty for correct and incorrect responses. These metrics are available without the additional raw-trace prefix instrumentation.

The final entropy sidecar is per example and per completed run. `answer_option_entropy` is computed over the normalized A/B/C/D option probabilities at the decoded answer position, so it captures uncertainty among the multiple-choice labels. `answer_option_raw_probabilities` are the model's raw probabilities for the candidate option tokens before renormalizing across A/B/C/D; these can be very different in scale from the displayed normalized option distribution. `decoded_token_entropy_mean`, median, and max summarize uncertainty over the whole generated answer text.

In [ ]:
final_entropy_rows = []
for entropy_path in sorted(OUTPUT_ROOT.glob("mined_by_lvar_ckpt/evaluated_by_lvar_ckpt/trace_variant_*/*_entropy_tracking.json")):
    model = "lvar"
    variant = entropy_path.parent.name.removeprefix("trace_variant_")
    trace_context = trace_context_from_output_path(entropy_path)
    replay_context = replay_context_from_output_path(entropy_path, trace_context)
    for row in json.loads(entropy_path.read_text()):
        if (trace_context, variant) not in EXPECTED_EXPERIMENTS:
            continue
        final_entropy_rows.append({
            "model": model, "trace_context": trace_context,
            "replay_context": replay_context, "variant": variant, **row,
        })

final_entropy_df = pd.DataFrame(final_entropy_rows)
if final_entropy_df.empty:
    print("No entropy sidecars found for the seven expected runs.")
else:
    final_entropy_df["variant"] = pd.Categorical(final_entropy_df["variant"], VARIANT_ORDER, ordered=True)
    final_entropy_summary = (
        final_entropy_df.groupby(["model", "trace_context", "replay_context", "variant", "correct"], observed=True)
        .agg(
            examples=("example_id", "nunique"),
            answer_option_entropy_mean=("answer_option_entropy", "mean"),
            decoded_token_entropy_mean=("decoded_token_entropy_mean", "mean"),
        )
        .reset_index()
    )
    display(final_entropy_summary.style.format({
        "answer_option_entropy_mean": "{:.4f}",
        "decoded_token_entropy_mean": "{:.4f}",
    }, na_rep="—").hide(axis="index"))

    if HAS_SEABORN:
        grid = sns.catplot(
            data=final_entropy_df, x="variant", y="answer_option_entropy",
            hue="correct", col="trace_context", row="replay_context", kind="box",
            sharey=True, height=3.2, aspect=1.25,
        )
        grid.set_axis_labels("Variant", "Final answer-option entropy (nats)")
        grid.set_titles("{col_name} / {row_name}")
        for axis in grid.axes.flat:
            axis.tick_params(axis="x", rotation=20)
        plt.show()